<a href="https://colab.research.google.com/github/asiyyajm/AJ-CSC_FinalProject/blob/main/AJ_CSC_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 321 Final Project
### Asiya Jaffer

*Abstract:* This project investigates how machine learning can be used to predict the severity of Parkinson's disease motor symptoms using voice measurements. Using the Parkinson's Telemonitoring dataset from the UCI Machine Learning Repository, which contains 5,875 recordings from 42 patients, we will examine how different voice features contribute to predicting motor UPDRS scores. We will compare several regression algorithms, identify the most important voice measurements, and investigate how removing or combining features affects prediction accuracy. We will also examine whether age, sex, and test time improve predictions. By analyzing these results, we aim to better understand which features are most useful for predicting Parkinson's motor symptom severity and how feature selection influences machine learning performance.

###Introduction
The goal of this project is to estimate the severity of Parkinson’s motor symptoms from measurements taken from voice recordings. The target is the motor score from the Unified Parkinson’s Disease Rating Scale (UPDRS). I will treat the task as a regression problem because this score is numerical. I will measure prediction error using root mean squared error (RMSE), reported in motor UPDRS points.
The dataset includes multiple recordings per patient, so I will keep each patient’s recordings together when splitting the data. This lets us test how well the models predict scores for patients that have not been seen yet. I will also investigate whether a smaller set of voice measurements can perform as well as the full set and whether patient characteristics or the timing of a recording improve predictions.


**Questions:**


1.   How accurately can voice measurements predict the severity of Parkinson's symptoms?
2.   Which voice measurements are the strongest predictors of Parkinson's symptom severity?


1.   How does selecting different groups of voice measurements affect the accuracy of predicting Parkinson's severity?
2.   How do age and gender compare to voice measurements in predicting Parkinson's symptoms severity?










###Motivation
Assessing Parkinson’s motor symptoms with the Unified Parkinson’s Disease Rating Scale (UPDRS) requires an examination by trained clinical staff. On the other hand, in order to gather information between visits, voice recordings can be collected at home. Through the Parkinson’s Telemonitoring dataset, measurements from these recordings are used to estimate UPDRS scores. This work motivates one of my questions: how well can voice measurements predict motor UPDRS scores for patients whose recordings were held out of training?

A 2020 study using a deep neural network on the same dataset reported a test RMSE of 1.42 motor UPDRS points. A 2026 study using an Extra Trees regressor and all 16 voice measurements reported a mean absolute error (MAE) of 7.21 motor UPDRS points when it held out one patient at a time. The second study also reported higher error under patient-level evaluation than under recording-level evaluation. The two error values use different metrics and evaluation setups, so a direct numerical comparison would be misleading.

In my project, I'll compare regression models with a ZeroR baseline that predicts the mean motor UPDRS score from the training data. On the same held-out patients, I'll examine how prediction error changes when we use individual voice measurements, compare jitter and shimmer feature groups with the full voice feature set and add age, sex, and test time. This lets us assess how much each set of inputs helps predict motor UPDRS scores for new patients.


###Data
I am going to be using the Parkinson’s Telemonitoring dataset from the UCI Machine Learning Repository. The CSV contains 5,875 rows and 22 columns. Each row contains measurements from one voice recording. The recordings were collected at home from 42 people with early-stage Parkinson’s disease during a six-month study; each participant contributed multiple recordings.

The file includes a participant ID, age, sex, time in days relative to recruitment, two UPDRS scores and 16 numerical voice measurements. These include five jitter measures that describe variation in voice frequency, six shimmer measures that describe variation in amplitude, and five other measures that capture properties such as the balance between noise and tone in the voice. Participant ages range from 36 to 85 years, and the dataset codes sex as 0 for male and 1 for female. My prediction target is motor_UPDRS, which ranges from 5.04 to 39.51 points. I will use subject# to keep each participant’s recordings together when splitting the data and exclude both subject# and total_UPDRS from the predictors.

After reviewing CSV, I found no missing values or exact duplicate rows. Additionally, the amount of recording noise is unknown at this stage.


###Experiments

In [2]:
import pandas as pd

parkinsons = pd.read_csv("parkinsons_updrs.csv")

print("Rows and columns:", parkinsons.shape)
print("Unique participants:", parkinsons["subject#"].nunique())
print("Missing values:", parkinsons.isna().sum().sum())
print("Exact duplicate rows:", parkinsons.duplicated().sum())

display(parkinsons[["age", "test_time", "motor_UPDRS"]].describe())

FileNotFoundError: [Errno 2] No such file or directory: 'parkinsons_updrs.csv'

In [1]:
from sklearn.model_selection import KFold

participant_ids = []

for participant_id in parkinsons["subject#"]:
    if participant_id not in participant_ids:
        participant_ids.append(participant_id)

folds = KFold(n_splits=10, shuffle=True, random_state=42)
participant_splits = list(folds.split(participant_ids))

print("Number of participants:", len(participant_ids))
print("Number of folds:", len(participant_splits))

NameError: name 'parkinsons' is not defined